# Photonics Signal IC Analysis
**Tickers:** AAOI, COHR, AEHR, LITE  
**Macro overlays:** VIX, SOX, TNX, /CL (via USO proxy), QQQ  
**Goal:** Compute per-signal Information Coefficient across multiple forward return horizons, identify optimal holding period per signal, cull redundant signals.

---

## 0. Install & Import

In [ ]:
!pip install yfinance pandas numpy scipy matplotlib seaborn -q

import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

## 1. Configuration

In [ ]:
# --- Tickers ---
# Core photonics universe (IPGP added as sector pure-play benchmark)
PHOTONICS   = ['AAOI', 'COHR', 'AEHR', 'LITE', 'IPGP']

# Macro overlay — yfinance symbol mapping:
#   TNX:CGI  → ^TNX   (CBOE 10Y rate)
#   /CLK26   → USO    (WTI crude proxy; continuous futures need broker API)
#   SOX      → ^SOX   (Philadelphia Semiconductor Index)
#   VIX      → ^VIX   (CBOE Volatility Index)
#   /ESM26   → ^GSPC  (S&P 500 spot; futures highly correlated)
#   SPX      → ^GSPC  (same underlying, deduplicated)
#   /NQ[M26] → QQQ    (Nasdaq futures proxy)
MACRO_TICKERS = ['^VIX', '^SOX', '^TNX', 'USO', '^GSPC', 'QQQ']
ALL_TICKERS   = PHOTONICS + MACRO_TICKERS

# --- Lookback for data pull ---
START = '2022-01-01'
END   = None   # None = today

# --- Forward return horizons to test (in trading days) ---
HORIZONS = [1, 3, 5, 10, 21]   # 1d, 3d, 1wk, 2wk, 1mo

# --- Signal lookback windows ---
MOMENTUM_WINDOWS  = [5, 10, 21]    # days
VOL_WINDOW        = 10             # days for volume ratio
SMA_WINDOW        = 200            # 200-day SMA
EMA_FAST          = 9
EMA_SLOW          = 21
BB_WINDOW         = 20             # Bollinger
BB_STD            = 2.0
VIX_THRESHOLD     = 20.0          # regime gate

print('Config set.')

## 2. Data Pull

In [ ]:
raw = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True)

close  = raw['Close'].copy()
volume = raw['Volume'].copy()
high   = raw['High'].copy()
low    = raw['Low'].copy()

# Rename macro columns to clean names
rename_map = {'^VIX': 'VIX', '^SOX': 'SOX', '^TNX': 'TNX', '^GSPC': 'SPX'}
close  = close.rename(columns=rename_map)
volume = volume.rename(columns=rename_map)
high   = high.rename(columns=rename_map)
low    = low.rename(columns=rename_map)

print(f'Data shape: {close.shape}')
print(f'Date range: {close.index[0].date()} to {close.index[-1].date()}')
close.tail(3)

## 3. Signal Construction

Each function returns a DataFrame aligned to the close index.  
Positive value = bullish signal, negative = bearish, zero = neutral.

In [ ]:
# ── Tier 1: Price / Technical Signals ──────────────────────────────────────

def sig_momentum(close, window):
    """n-day price momentum (return over lookback)"""
    return close[PHOTONICS].pct_change(window)

def sig_volume_surge(volume, close, window=VOL_WINDOW):
    """Current volume / rolling mean volume - 1. >0 = above avg."""
    avg_vol = volume[PHOTONICS].rolling(window).mean()
    return (volume[PHOTONICS] / avg_vol) - 1

def sig_sma200_distance(close):
    """% distance of price from 200-day SMA. Positive = above SMA."""
    sma = close[PHOTONICS].rolling(SMA_WINDOW).mean()
    return (close[PHOTONICS] - sma) / sma

def sig_ema_cross(close, fast=EMA_FAST, slow=EMA_SLOW):
    """Fast EMA minus slow EMA normalized by price. Positive = fast above slow."""
    ema_f = close[PHOTONICS].ewm(span=fast).mean()
    ema_s = close[PHOTONICS].ewm(span=slow).mean()
    return (ema_f - ema_s) / close[PHOTONICS]

def sig_bollinger_position(close, window=BB_WINDOW, n_std=BB_STD):
    """%B: where price sits in the Bollinger band. 0=lower, 1=upper, 0.5=mid."""
    mid  = close[PHOTONICS].rolling(window).mean()
    std  = close[PHOTONICS].rolling(window).std()
    upper = mid + n_std * std
    lower = mid - n_std * std
    pct_b = (close[PHOTONICS] - lower) / (upper - lower)
    return pct_b - 0.5   # center so 0 = midline

# ── Tier 2: Macro Signals ──────────────────────────────────────────────────

def sig_vix_regime(close):
    """Binary: +1 if VIX < threshold (risk-on), -1 if above."""
    vix = close['VIX']
    regime = pd.Series(np.where(vix < VIX_THRESHOLD, 1, -1), index=vix.index)
    # Broadcast to all photonics tickers
    return pd.DataFrame({t: regime for t in PHOTONICS})

def sig_vix_change(close, window=1):
    """1-day VIX return (negative = VIX falling = bullish). Inverted."""
    vix_ret = -close['VIX'].pct_change(window)
    return pd.DataFrame({t: vix_ret for t in PHOTONICS})

def sig_sox_relative(close):
    """Photonics ticker 1d return minus SOX 1d return. Positive = outperforming."""
    sox_ret  = close['SOX'].pct_change(1)
    tick_ret = close[PHOTONICS].pct_change(1)
    return tick_ret.subtract(sox_ret, axis=0)

def sig_tnx_change(close, window=1):
    """Rate change (inverted — rising rates = headwind for growth)."""
    tnx_ret = -close['TNX'].pct_change(window)
    return pd.DataFrame({t: tnx_ret for t in PHOTONICS})

def sig_oil_change(close, window=1):
    """USO (oil proxy for /CLK26) 1-day return. Let IC decide direction for photonics."""
    oil_ret = close['USO'].pct_change(window)
    return pd.DataFrame({t: oil_ret for t in PHOTONICS})

def sig_spx_relative(close):
    """Photonics ticker 1d return minus SPX 1d return. Positive = outperforming broad market."""
    spx_ret  = close['SPX'].pct_change(1)
    tick_ret = close[PHOTONICS].pct_change(1)
    return tick_ret.subtract(spx_ret, axis=0)

def sig_sox_vs_spx(close):
    """SOX return minus SPX return. Positive = semis leading market. Sector momentum signal."""
    sox_ret = close['SOX'].pct_change(1)
    spx_ret = close['SPX'].pct_change(1)
    spread  = sox_ret - spx_ret
    return pd.DataFrame({t: spread for t in PHOTONICS})

print('Signal functions defined.')

In [ ]:
# Build signal dictionary
signals = {
    # Tier 1
    'Momentum_5d'       : sig_momentum(close, 5),
    'Momentum_10d'      : sig_momentum(close, 10),
    'Momentum_21d'      : sig_momentum(close, 21),
    'Volume_Surge'      : sig_volume_surge(volume, close),
    'SMA200_Distance'   : sig_sma200_distance(close),
    'EMA_Cross_9_21'    : sig_ema_cross(close),
    'Bollinger_Pct_B'   : sig_bollinger_position(close),
    # Tier 2
    'VIX_Regime'        : sig_vix_regime(close),
    'VIX_Change_1d'     : sig_vix_change(close),
    'SOX_Relative'      : sig_sox_relative(close),
    'TNX_Change'        : sig_tnx_change(close),
    'Oil_Change'        : sig_oil_change(close),
    # SPX-based
    'SPX_Relative'      : sig_spx_relative(close),
    'SOX_vs_SPX'        : sig_sox_vs_spx(close),
}

print(f'{len(signals)} signals built:')
for k in signals:
    print(f'  {k}')

## 4. IC Calculation

For each signal × ticker × horizon:  
IC = Spearman rank correlation(signal[t], forward_return[t+h])

We use Spearman (rank-based) because it's more robust to outliers than Pearson — standard institutional practice.

In [ ]:
def compute_ic(signal_df, close, ticker, horizon):
    """Compute Spearman IC for one signal × ticker × horizon."""
    fwd_ret = close[ticker].pct_change(horizon).shift(-horizon)
    sig     = signal_df[ticker]
    combined = pd.concat([sig, fwd_ret], axis=1).dropna()
    combined.columns = ['signal', 'fwd_ret']
    if len(combined) < 30:
        return np.nan, np.nan
    ic, pval = spearmanr(combined['signal'], combined['fwd_ret'])
    return ic, pval

# Build results table
rows = []
for sig_name, sig_df in signals.items():
    for ticker in PHOTONICS:
        for h in HORIZONS:
            ic, pval = compute_ic(sig_df, close, ticker, h)
            rows.append({
                'Signal'  : sig_name,
                'Ticker'  : ticker,
                'Horizon' : h,
                'IC'      : ic,
                'p_value' : pval,
                'Sig_05'  : pval < 0.05 if not np.isnan(pval) else False
            })

ic_df = pd.DataFrame(rows)
print(f'IC table shape: {ic_df.shape}')
ic_df.head(10)

## 5. Optimal Horizon Per Signal

For each signal, find the horizon where mean |IC| across tickers is highest.

In [ ]:
# Mean IC and |IC| across tickers, grouped by signal × horizon
summary = (
    ic_df
    .groupby(['Signal', 'Horizon'])
    .agg(
        Mean_IC   = ('IC', 'mean'),
        Mean_AbsIC= ('IC', lambda x: x.abs().mean()),
        Pct_Sig   = ('Sig_05', 'mean')   # fraction of tickers with p<0.05
    )
    .reset_index()
)

# Best horizon per signal = argmax Mean_AbsIC
best_horizon = (
    summary
    .loc[summary.groupby('Signal')['Mean_AbsIC'].idxmax()]
    .rename(columns={'Horizon': 'Optimal_Horizon'})
    .sort_values('Mean_AbsIC', ascending=False)
    .reset_index(drop=True)
)

print('=== Signals ranked by peak |IC| ===')
print(best_horizon[['Signal','Optimal_Horizon','Mean_IC','Mean_AbsIC','Pct_Sig']].to_string(index=False))

## 6. IC Heatmap — All Signals × All Tickers (at optimal horizon)

In [ ]:
# Pivot: signal × ticker at each signal's optimal horizon
opt_h_map = best_horizon.set_index('Signal')['Optimal_Horizon'].to_dict()

heatmap_rows = []
for sig_name in signals:
    h = opt_h_map[sig_name]
    for ticker in PHOTONICS:
        ic, pval = compute_ic(signals[sig_name], close, ticker, h)
        heatmap_rows.append({'Signal': sig_name, 'Ticker': ticker, 'IC': ic})

heatmap_df = pd.DataFrame(heatmap_rows).pivot(index='Signal', columns='Ticker', values='IC')

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    heatmap_df,
    annot=True, fmt='.3f',
    cmap='RdYlGn', center=0,
    vmin=-0.3, vmax=0.3,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Information Coefficient at Optimal Horizon\n(Spearman, each signal × ticker)', fontsize=12, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('ic_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ic_heatmap.png')

## 7. IC Curve — Signal Strength Across Horizons

In [ ]:
n_sigs = len(signals)
n_cols = 4
n_rows = (n_sigs + n_cols - 1) // n_cols  # dynamic row count
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5), sharey=True)
axes = axes.flatten()

for idx, sig_name in enumerate(signals):
    ax = axes[idx]
    for ticker in PHOTONICS:
        ic_vals = []
        for h in HORIZONS:
            ic, _ = compute_ic(signals[sig_name], close, ticker, h)
            ic_vals.append(ic)
        ax.plot(HORIZONS, ic_vals, marker='o', label=ticker, linewidth=1.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.axhline(0.05, color='green', linewidth=0.5, linestyle=':')
    ax.axhline(-0.05, color='red', linewidth=0.5, linestyle=':')
    ax.set_title(sig_name.replace('_', ' '), fontsize=8)
    ax.set_xticks(HORIZONS)
    ax.set_xlabel('Horizon (days)', fontsize=7)
    ax.set_ylabel('IC', fontsize=7)
    if idx == 0:
        ax.legend(fontsize=6)

# Hide unused subplots
for j in range(len(signals), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('IC Curve by Signal × Ticker Across Holding Period Horizons', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ic_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ic_curves.png')

## 8. Signal Correlation Matrix (Independence Check)

High correlation between signals = redundant = candidate to cull.

In [ ]:
# Stack all signal values across tickers into one pooled series per signal
sig_corr_data = {}
for sig_name, sig_df in signals.items():
    # Stack all tickers — tests if signals move together across the universe
    pooled = sig_df[PHOTONICS].stack().reset_index(drop=True)
    sig_corr_data[sig_name] = pooled

sig_corr_df = pd.DataFrame(sig_corr_data).dropna()
corr_matrix = sig_corr_df.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Signal Correlation Matrix (Spearman)\nHigh values = redundant signals', fontsize=12)
plt.tight_layout()
plt.savefig('signal_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: signal_correlation.png')

## 9. Information Ratio — Combined Stack

IR = IC_mean × √N_effective  
N_effective accounts for signal correlations.

In [ ]:
# Mean |IC| per signal (at optimal horizon, averaged across tickers)
ic_mean_per_signal = best_horizon.set_index('Signal')['Mean_AbsIC']

# Effective N from correlation matrix
# N_eff = N^2 / sum(corr_matrix^2)  — standard formula
N = len(signals)
corr_sq_sum = (corr_matrix.values ** 2).sum()
N_eff = (N ** 2) / corr_sq_sum

IC_avg = ic_mean_per_signal.mean()
IR_naive    = IC_avg * np.sqrt(N)
IR_adjusted = IC_avg * np.sqrt(N_eff)

print('=== Information Ratio Summary ===')
print(f'Signals in stack     : {N}')
print(f'Effective independent: {N_eff:.2f}')
print(f'Mean |IC|            : {IC_avg:.4f}')
print(f'IR (naive √N)        : {IR_naive:.4f}')
print(f'IR (adjusted N_eff)  : {IR_adjusted:.4f}')
print()
print('Rule of thumb: IR > 0.5 = strong edge | 0.3-0.5 = decent | <0.3 = weak')

## 10. Culling Recommendations

In [ ]:
# Cull criteria:
# 1. Mean |IC| < 0.03  (too weak)
# 2. No tickers with p<0.05  (never statistically significant)
# 3. Correlation > 0.7 with a stronger signal (redundant)

IC_THRESHOLD   = 0.03
CORR_THRESHOLD = 0.70

cull_list    = []
keep_list    = []
review_list  = []

ranked_signals = best_horizon['Signal'].tolist()   # sorted by IC descending

for sig in ranked_signals:
    row = best_horizon[best_horizon['Signal'] == sig].iloc[0]
    ic_val  = row['Mean_AbsIC']
    pct_sig = row['Pct_Sig']

    # Check redundancy with any kept signal
    redundant_with = None
    for kept in keep_list:
        if abs(corr_matrix.loc[sig, kept]) > CORR_THRESHOLD:
            redundant_with = kept
            break

    if ic_val < IC_THRESHOLD:
        cull_list.append((sig, 'IC too weak', ic_val))
    elif pct_sig == 0:
        cull_list.append((sig, 'Never significant (p>0.05)', ic_val))
    elif redundant_with:
        review_list.append((sig, f'Correlated with {redundant_with}', ic_val))
    else:
        keep_list.append(sig)

print('=== KEEP ===')
for s in keep_list:
    row = best_horizon[best_horizon['Signal']==s].iloc[0]
    print(f'  {s:<25} IC={row["Mean_AbsIC"]:.4f}  OptH={int(row["Optimal_Horizon"])}d')

print('\n=== REVIEW (possibly redundant) ===')
for s, reason, ic in review_list:
    print(f'  {s:<25} IC={ic:.4f}  Reason: {reason}')

print('\n=== CULL ===')
for s, reason, ic in cull_list:
    print(f'  {s:<25} IC={ic:.4f}  Reason: {reason}')

## 11. Export Results to CSV

In [ ]:
# Full IC table
ic_df.to_csv('ic_full_results.csv', index=False)

# Summary by signal
best_horizon.to_csv('ic_signal_summary.csv', index=False)

# Correlation matrix
corr_matrix.to_csv('signal_correlation_matrix.csv')

print('Exports complete:')
print('  ic_full_results.csv')
print('  ic_signal_summary.csv')
print('  signal_correlation_matrix.csv')
print('  ic_heatmap.png')
print('  ic_curves.png')
print('  signal_correlation.png')
print()
print('Upload ic_signal_summary.csv to Claude for final factor strength document.')